## Supervised Model - Regression Approach 

#### By Amina Jobarteh

#### Purpose of Regression Model

Leomary and Oluwafikunayomi created a clustering technique to group the fatal accidents data into 5 clusters (such as *Fast Rural Roads* or *The Morning Commute*). The clustering tool did not know about fatalities when it made these groups. Even so, it naturally separated safe driving situations from deadly ones. 

By feeding these cluster labels into our supervised regression model, we are giving it a helpful shortcut. Instead of forcing the model to figure out complex road combinations on its own, we hand it 5 profiles. The regression model then uses these profiles, along with our other key features, to calculate the exact mathematical probability (from 0% to 100%) of a crash being fatal.

- Cluster 0 (Fast Rural Roads): High-speed (62 mph) country roads during daylight. This is the most dangerous environment with a 2.65% actual fatality rate.

- Cluster 3 (Multi-Vehicle Collisions): Pile-ups involving an average of 3.4 cars on 42 mph roads. This is a high-risk environment with a 1.94% fatality rate.

- Cluster 4 (Urban After Dark): Town roads (30 mph) at intersections around 7:30 PM with streetlights on. This carries a moderate risk of 1.03%.

- Cluster 1 (The Morning Commute): Town roads (30 mph) at intersections around 7:00 AM in daylight. This carries a low-to-moderate risk of 0.98%.Cluster 2 (Urban Afternoon): The exact same town roads as Cluster 1, but around 3:00 PM. This is the largest group and the safest environment with a 0.57% fatality rate.

In [2]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

To fulfill our project goals, we tested two different advanced machine learning approaches to predict continuous accident risk and compared them using standard regression metrics.

- Approach 1: HistGradientBoostingRegressor (Our baseline tree model)

- Approach 2: RandomForestRegressor (An ensemble model using 100 decision trees)

### Aproach 1 HistGradientBoostingResgressor 

## Section 1 
We load the main dataset and our cluster dataset. We merge them together using the unique Accident_Index column. This brings our raw accident features and our new cluster profiles into one single dataframe.

In [3]:
df = pd.read_csv("../data/preprocessed/accidents_clean.csv")
clusters = pd.read_csv("../data/preprocessed/accident_clusters.csv.gz")

# Merge them together on the unique accident ID
df = df.merge(clusters, on="Accident_Index", how="left")

/var/folders/fy/syjpkkbx547fsjc43rbzh31m0000gn/T/ipykernel_17830/3898626110.py:1: DtypeWarning: Columns (0: Accident_Index) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/preprocessed/accidents_clean.csv")


## Section 2

We define what we want to predict (y) and what clues we will use to predict it (X).Our target (y) is a 1 if the accident was fatal and a 0 if the person survived.Our features (X) include the 5 cluster profiles plus the exact Hour, Speed_limit, and Number_of_Vehicles as recommended by our findings.

In [4]:
y = (df["Accident_Severity"] == "Fatal").astype(int)

feature_cols = [
    "cluster",
    "Number_of_Vehicles",
    "Hour",  
    "Speed_limit",  
    "Urban_or_Rural_Area",
    "Road_Type",
    "Light_Conditions",
]
X = df[feature_cols]

## Section 3
Used one-hot encoding (pd.get_dummies) to turn all of our text categories and cluster numbers into columns of 1s and 0s.

In [5]:

X = pd.get_dummies(
    X,
    columns=[
        "cluster",
        "Urban_or_Rural_Area",
        "Road_Type",
        "Light_Conditions",
    ],
    drop_first=True,
)


## Section 4 

We split our massive 1.9+ million rows into a training set and a testing set. We use the exact split from our findings: 1,593,245 rows to train the model, and 398,312 rows saved to test it.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=398312, random_state=42, stratify=y
)

## Section 5

Because our dataset is so huge, standard models can freeze the computer. We train a HistGradientBoostingRegressor. This model is built to handle millions of rows and categorical splits instantly.

In [7]:

regressor = HistGradientBoostingRegressor(random_state=42)

regressor.fit(X_train, y_train)

,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide ` and:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_categorical.py`... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 

## Section 6 

We run a test to see which features our model relies on the most. This confirms if our engineered clusters actually help the model make better guesses, or if the raw numbers do all the heavy lifting.

Does the cluster label actually help the supervised model? We run a Permutation Importance test to see how much the model's accuracy drops if we take the clusters away.

In [12]:
importance = permutation_importance(
    regressor, X_test, y_test, n_repeats=3, random_state=42
)

for i in importance.importances_mean.argsort()[::-1]:
    print(f"{X.columns[i]}: {importance.importances_mean[i]:.5f}")

Speed_limit: 0.01261
Hour: 0.00593
Number_of_Vehicles: 0.00475
Road_Type_Single carriageway: 0.00209
Urban_or_Rural_Area_Urban: 0.00197
Light_Conditions_Daylight: 0.00082
Road_Type_Roundabout: 0.00076
Light_Conditions_Darkness - no lighting: 0.00059
cluster_3.0: 0.00030
Light_Conditions_Darkness - lights lit: 0.00015
Road_Type_Slip road: 0.00012
Road_Type_One way street: 0.00004
cluster_1.0: 0.00004
Light_Conditions_Darkness - lights unlit: 0.00001
cluster_4.0: 0.00000
Urban_or_Rural_Area_Unallocated: 0.00000
cluster_2.0: 0.00000


We trained a supervised model (HistGradientBoostingRegressor) to look at individual accidents and predict their exact danger level. When we ask the model what clues it relies on most, it outputs a scoreboard of weights:

- Speed_limit (0.01261), Hour (0.00593), and Number_of_Vehicles (0.00475) are the undisputed heavy hitters. The model looks at these raw numbers first to make its decisions.

- The Cluster Labels (like cluster_3.0) scored very low or even 0.00000.

What this means: The model can predict well based on the data we fed it. It doesn't need to lean on our pre-made cluster "labels" because it uses the raw ingredients (speed, time, and car counts) to figure out the danger on its own.

## Section 7 

We predict the risk percentage for every accident in our test set. Then, we group the results by our 5 original cluster profiles to compare the model's average predicted risk against what actually happened in real life.

You might notice a tiny difference between the original clustering team's numbers and our validation scores. For example, Cluster 0 drops from a 2.65% fatal rate down to 2.41%.This is intentional and proof of a proper setup:

- To test our model fairly, we sliced our data. We hid 20% of the accidents in a secret drawer (the Test Set) and never let the model see them while it was learning.

- Because this Test Set is a smaller, random sample of the real world, its actual accident numbers fluctuate slightly due to pure random chance.

- This proves we are testing our model with brand-new data it has never memorized.

In [ ]:
X_test_analysis = X_test.copy()
X_test_analysis["predicted_risk"] = regressor.predict(X_test)
X_test_analysis["actual_fatal"] = y_test

cluster_cols = [col for col in X_test_analysis.columns if "cluster_" in col]


X_test_analysis["original_cluster"] = "cluster_0.0"


has_cluster = X_test_analysis[cluster_cols].sum(axis=1) > 0
X_test_analysis.loc[has_cluster, "original_cluster"] = X_test_analysis[
    cluster_cols
].idxmax(axis=1)

full_summary = X_test_analysis.groupby("original_cluster")[
    ["actual_fatal", "predicted_risk"]
].mean()
print(full_summary)

                  actual_fatal  predicted_risk
original_cluster                              
cluster_0.0           0.024119        0.023804
cluster_1.0           0.009821        0.009676
cluster_2.0           0.005459        0.005877
cluster_3.0           0.018655        0.019077
cluster_4.0           0.010752        0.010039


The ultimate test of our project is seeing if our model's predictions match the real-world outcomes in our hidden test set. When we group our model's predictions by the five environments, the results are nearly identical.

Even though the regression model focuses on raw numbers like speed limits rather than cluster names, it shows close accuracy with the risk profiles we determined in the clustering.

Interpretations: 

- The predicted risk percentages land close to the actual fatality rates for every single profile.

- Speed_limit, Hour, and Number_of_Vehicles are the model's most important features. This proves our team's recommendation to include them was correct. This information will be invaluable to stakeholders as they can increase speed limit sign reminders on the road at certain times. Also increase road safety signs or signals on the road during hours that see traffic patterns.

- Cluster 0 (Fast Rural Roads): The model successfully learned this is the deadliest group. It predicted a 2.38% fatality risk, which is close to the actual 2.41% rate in the data. This can signal to stakeholders possible reduction in speed limit on these rural roads, or increase in vehichle awareness when using fast rural roads. 

- Cluster 2 (Urban Afternoons): The model correctly identified this as the safest profile, predicting a very low 0.58% risk against the actual 0.54% rate. This can also signal to stakeholders that the time of day contributes to the likelihood of accidents. Accidents can still happen in urban afternoons, so speed limit reminders on the road may still benefit, but since we see this as the safest cluster, this can turn their heads to look at other profiles like Urban after dark, which may have a higher fatal rate. 

- Even though individual cluster importance scores look small, they act as vital structural anchors. They help the model map out the massive 4.4x jump in danger between urban afternoons and fast rural roads.

### Approach 2 - Random Forest Regression

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

rf_model = make_pipeline(
    SimpleImputer(strategy='median'),
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
)


rf_model.fit(X_train, y_train)


rf_preds = rf_model.predict(X_test)


In [ ]:
X_test_rf = X_test.copy()
X_test_rf["predicted_risk"] = rf_preds
X_test_rf["actual_fatal"] = y_test

cluster_cols = [col for col in X_test_rf.columns if "cluster_" in col]
X_test_rf["original_cluster"] = "cluster_0.0"
has_cluster = X_test_rf[cluster_cols].sum(axis=1) > 0
X_test_rf.loc[has_cluster, "original_cluster"] = X_test_rf[cluster_cols].idxmax(axis=1)

rf_summary = X_test_rf.groupby("original_cluster")[["actual_fatal", "predicted_risk"]].mean()
print(rf_summary)

                  actual_fatal  predicted_risk
original_cluster                              
cluster_0.0           0.024119        0.023873
cluster_1.0           0.009821        0.009888
cluster_2.0           0.005459        0.005767
cluster_3.0           0.018655        0.020172
cluster_4.0           0.010752        0.010143


Random Forest model accurately captures the risk profiles of each cluster.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("--- 1. HistGradientBoostingRegressor Metrics ---")
print(f"MAE:  {mean_absolute_error(y_test, X_test_analysis['predicted_risk']):.5f}")
print(f"MSE:  {mean_squared_error(y_test, X_test_analysis['predicted_risk']):.5f}")
print(f"R2:   {r2_score(y_test, X_test_analysis['predicted_risk']):.5f}")
print("\n")

print("--- 2. RandomForestRegressor Metrics ---")
print(f"MAE:  {mean_absolute_error(y_test, rf_preds):.5f}")
print(f"MSE:  {mean_squared_error(y_test, rf_preds):.5f}")
print(f"R2:   {r2_score(y_test, rf_preds):.5f}")

--- 1. HistGradientBoostingRegressor Metrics ---
MAE:  0.02500
MSE:  0.01252
R2:   0.01551


--- 2. RandomForestRegressor Metrics ---
MAE:  0.02508
MSE:  0.01267
R2:   0.00308


In [ ]:
# For Tableau
full_summary.to_csv("../data/preprocessed/regress1hgb_tableau_results.csv")
rf_summary.to_csv("../data/preprocessed/regress2rf_tableau_results.csv")


Analysis and Insights:

- Both models performed well in terms of average error. The HistGradientBoosting model made mistakes in overall risk by an average of just 2.50% (MAE = 0.02500), slightly beating out the RandomForest model's 2.51%.

- The HistGradientBoosting model achieved an \(R^{2}\) score of 0.01551, which is roughly 5 times higher than the RandomForest score of 0.00308. While both scores are low—reflecting the unpredictable nature of individual traffic accidents, both agree on how dangerous the groups are. But HistGradientBoosting is better at adjusting its guesses up and down for individual rows, while Random Forest protects itself from random chaos by leaning heavily on group averages.

- When grouped by cluster, the Random Forest predicted a 2.38% risk for the high-danger environment (Cluster 0, actual 2.41%) and a 0.57% risk for the safest environment (Cluster 2, actual 0.54%). This proves that our unsupervised feature engineering was stable, allowing both models to reconstruct the cluster profiles.

## Transition to Morris's Analysis

Amina completed the first part of the supervised modeling using a regression approach. Building on her work, I continued with classification models because our target is whether an accident was fatal or not fatal.

In Sections 8 through 12, I will test two additional classification models, compare all three approaches using appropriate performance metrics, identify the strongest model, and interpret the results.

## Section 8 — Logistic Regression

#### By Morris

I started with Logistic Regression as a simple classification model. It gives me a baseline for predicting whether an accident was fatal or not fatal. I used the same training and testing data and the same features from the previous analysis so the models can be compared fairly.

In [ ]:
# Section 8: Logistic Regression
# Classification model

from sklearn.linear_model import LogisticRegression

# Make copies so Amina's data is not changed
X_train_logistic = X_train.copy()
X_test_logistic = X_test.copy()

# Fill missing values using the training-set median
X_train_logistic = X_train_logistic.fillna(X_train_logistic.median())
X_test_logistic = X_test_logistic.fillna(X_train_logistic.median())

print("Missing values in training data:", X_train_logistic.isna().sum().sum())
print("Missing values in testing data:", X_test_logistic.isna().sum().sum())

# Train Logistic Regression
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_logistic, y_train)

print("Logistic Regression model trained successfully.")

Missing values in training data: 0
Missing values in testing data: 0
Logistic Regression model trained successfully.


## Section 9 — Logistic Regression Evaluation

After training the Logistic Regression model, I evaluated its performance on the testing data. Because fatal accidents are much less common than non-fatal accidents, I used precision, recall, F1 score, and ROC-AUC in addition to accuracy. These metrics give a better picture of how well the model identifies the less common fatal accidents.


In [ ]:
# Section 9: Evaluate Logistic Regression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Make predictions on the testing data
logistic_pred = logistic_model.predict(X_test_logistic)
logistic_prob = logistic_model.predict_proba(X_test_logistic)[:, 1]

# Calculate performance metrics
logistic_accuracy = accuracy_score(y_test, logistic_pred)
logistic_precision = precision_score(y_test, logistic_pred, zero_division=0)
logistic_recall = recall_score(y_test, logistic_pred, zero_division=0)
logistic_f1 = f1_score(y_test, logistic_pred, zero_division=0)
logistic_roc_auc = roc_auc_score(y_test, logistic_prob)

print("Logistic Regression Results")
print("---------------------------")
print(f"Accuracy:  {logistic_accuracy:.4f}")
print(f"Precision: {logistic_precision:.4f}")
print(f"Recall:    {logistic_recall:.4f}")
print(f"F1 Score:  {logistic_f1:.4f}")
print(f"ROC-AUC:   {logistic_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, logistic_pred))

Logistic Regression Results
---------------------------
Accuracy:  0.9871
Precision: 0.0000
Recall:    0.0000
F1 Score:  0.0000
ROC-AUC:   0.7387

Confusion Matrix:
[[393182      0]
 [  5130      0]]


The Logistic Regression model achieved 98.71% accuracy, but accuracy alone does not tell the full story. The model did not correctly identify any fatal accidents, resulting in 0.00 precision, recall, and F1 score. This is mainly because fatal accidents are much less common than non-fatal accidents in the dataset.

The ROC-AUC score of 0.7387 shows that the model still has some ability to distinguish between fatal and non-fatal accidents based on the predicted probabilities. However, its default classification threshold resulted in no accidents being classified as fatal. This shows why looking at multiple evaluation metrics is important for this project.


## Section 10 — HistGradientBoosting Classifier

After evaluating Logistic Regression, I tested a second classification model using HistGradientBoostingClassifier. This model can capture more complex relationships between the accident features than a simple linear model. I used the same training and testing data so the results can be compared fairly.


In [ ]:
# Section 10: HistGradientBoosting Classifier

from sklearn.ensemble import HistGradientBoostingClassifier

# Make copies so the original data is not changed
X_train_hgb = X_train.copy()
X_test_hgb = X_test.copy()

# Fill missing values using the training-set median
X_train_hgb = X_train_hgb.fillna(X_train_hgb.median())
X_test_hgb = X_test_hgb.fillna(X_train_hgb.median())

# Create the model
hgb_classifier = HistGradientBoostingClassifier(
    random_state=42
)

# Train the model
hgb_classifier.fit(X_train_hgb, y_train)

print("HistGradientBoostingClassifier trained successfully.")

HistGradientBoostingClassifier trained successfully.


In [ ]:
# Evaluate HistGradientBoostingClassifier

hgb_pred = hgb_classifier.predict(X_test_hgb)
hgb_prob = hgb_classifier.predict_proba(X_test_hgb)[:, 1]

hgb_accuracy = accuracy_score(y_test, hgb_pred)
hgb_precision = precision_score(y_test, hgb_pred, zero_division=0)
hgb_recall = recall_score(y_test, hgb_pred, zero_division=0)
hgb_f1 = f1_score(y_test, hgb_pred, zero_division=0)
hgb_roc_auc = roc_auc_score(y_test, hgb_prob)

print("HistGradientBoostingClassifier Results")
print("--------------------------------------")
print(f"Accuracy:  {hgb_accuracy:.4f}")
print(f"Precision: {hgb_precision:.4f}")
print(f"Recall:    {hgb_recall:.4f}")
print(f"F1 Score:  {hgb_f1:.4f}")
print(f"ROC-AUC:   {hgb_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, hgb_pred))

HistGradientBoostingClassifier Results
--------------------------------------
Accuracy:  0.9871
Precision: 0.3333
Recall:    0.0002
F1 Score:  0.0004
ROC-AUC:   0.7717

Confusion Matrix:
[[393180      2]
 [  5129      1]]


The HistGradientBoostingClassifier achieved 98.71% accuracy, which is the same accuracy as the Logistic Regression model. However, its ROC-AUC increased from 0.7387 with Logistic Regression to 0.7717. This suggests that the HistGradientBoostingClassifier is better at distinguishing between fatal and non-fatal accidents based on its predicted probabilities.

The model identified one fatal accident, giving it a precision of 33.33%. However, its recall was only 0.02%, meaning it still missed almost all of the fatal accidents in the test set. This shows that accuracy is still not a good measure of success for this highly imbalanced target.


## Section 11 — Comparing the Supervised Models

I compared the two classification models using accuracy, precision, recall, F1 score, and ROC-AUC. Because fatal accidents are much less common than non-fatal accidents, I focused especially on recall, F1 score, and ROC-AUC rather than relying on accuracy alone.

The comparison also helps show whether the more complex HistGradientBoostingClassifier provides an improvement over the simpler Logistic Regression model.


In [ ]:
# Section 11: Compare the classification models

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "HistGradientBoostingClassifier"
    ],
    "Accuracy": [
        logistic_accuracy,
        hgb_accuracy
    ],
    "Precision": [
        logistic_precision,
        hgb_precision
    ],
    "Recall": [
        logistic_recall,
        hgb_recall
    ],
    "F1 Score": [
        logistic_f1,
        hgb_f1
    ],
    "ROC-AUC": [
        logistic_roc_auc,
        hgb_roc_auc
    ]
})

comparison.round(4)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.9871,0.0000,0.0000,0.0000,0.7387
1,HistGradientBoostingClassifier,0.9871,0.3333,0.0002,0.0004,0.7717


## Section 12 — Final Model Selection and Findings

Based on the comparison, HistGradientBoostingClassifier performed better than Logistic Regression. Both models had the same accuracy of 98.71%, but HistGradientBoostingClassifier achieved a higher ROC-AUC of 0.7717 compared with 0.7387 for Logistic Regression. It also identified one fatal accident, while Logistic Regression did not identify any.

However, both models had very low recall because fatal accidents are rare compared with non-fatal accidents. This means neither model is strong enough to identify fatal accidents using the default classification threshold.

For this analysis, I would select HistGradientBoostingClassifier as the stronger classification approach because it provided better overall separation between fatal and non-fatal accidents based on ROC-AUC. At the same time, the results show that further work would be needed before using the model for real-world fatality-risk prediction, such as addressing class imbalance and adjusting the classification threshold.


In [ ]:
# Section 12: Final model comparison

best_model = comparison.loc[
    comparison["ROC-AUC"].idxmax()
]

print("Best classification model based on ROC-AUC:")
print(best_model)

Best classification model based on ROC-AUC:
Model        HistGradientBoostingClassifier
Accuracy                           0.987118
Precision                          0.333333
Recall                             0.000195
F1 Score                            0.00039
ROC-AUC                            0.771733
Name: 1, dtype: object


### Class Imbalance Limitation

One important limitation is the large difference between fatal and non-fatal accidents in the dataset. Only about 1.3% of the accidents were fatal. Because of this imbalance, both classification models achieved high accuracy while still identifying very few fatal accidents.

For this reason, accuracy alone should not be used to judge the models. Recall, F1 score, and ROC-AUC provide a more useful view of how well the models distinguish fatal accidents from non-fatal accidents.
